In [ ]:
# Train Qwen2.5-Math-1.5B as a Process Reward Model (PRM) on PRM800K — Phase 1 on kaggle

# %% CELL 1 — Installing dependencies
!pip install -q -U transformers accelerate peft bitsandbytes datasets

In [3]:
!pip install -U torchao #Additional requirements

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.2 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [6]:
# %% CELL 2 — Imports
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # reduce alloc fragmentation

import random
from dataclasses import dataclass
from typing import List, Dict, Any

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

MODEL_ID = "Qwen/Qwen2.5-Math-1.5B"
MAX_LENGTH_CAP = 1024              # hard ceiling; actual MAX_LENGTH is set adaptively in CELL 6
INCLUDE_NEUTRAL_AS_NEGATIVE = False  # rating==0 ("neutral") handling, see CELL 4
USE_4BIT = False   # 1.5B fits comfortably in fp16 on a single T4; skip quant noise
GRADIENT_CHECKPOINTING = True   # single source of truth, used by both CELL 8 and CELL 9

# IMPORTANT: this is plain text, NOT a special vocab token. We only ever read
# logits from the LAST position of the tokenized prompt, so it doesn't matter
# how many sub-word tokens this string breaks into -- no vocab resize needed.
STEP_TAG = "\nStep verdict:"

OUTPUT_DIR = "/kaggle/working/137qwen25math15b-prm-phase1"

In [7]:
# %% CELL 3 — Load PRM800K phase 1 from the Hub
from datasets import load_dataset, DatasetDict

phase1_train = load_dataset(
    "json",
    data_files={
        "train":
        "https://huggingface.co/datasets/tasksource/PRM800K/resolve/main/phase1_train.jsonl"
    }
)

phase1_test = load_dataset(
    "json",
    data_files={
        "test":
        "https://huggingface.co/datasets/tasksource/PRM800K/resolve/main/phase1_test.jsonl"
    }
)

# Combine into a single DatasetDict
raw = DatasetDict({
    "train": phase1_train["train"],
    "test": phase1_test["test"],
})

print(raw)
# DatasetDict({ train: Dataset({...949 rows}), test: Dataset({...106 rows}) })

phase1_train.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

phase1_test.jsonl: 0.00B [00:00, ?B/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labeler', 'timestamp', 'generation', 'is_quality_control_question', 'is_initial_screening_question', 'question', 'label'],
        num_rows: 949
    })
    test: Dataset({
        features: ['labeler', 'timestamp', 'generation', 'is_quality_control_question', 'is_initial_screening_question', 'question', 'label'],
        num_rows: 106
    })
})


In [12]:
# %% CELL 4 — Turn raw PRM800K rows into flat (text, label) classification examples
#
# PRM800K phase-1 schema per row:
#   question = {"problem": str, "ground_truth_answer": str}
#   label = {
#     "steps": [
#        {
#          "completions": [{"text": str, "rating": -1|0|1, "flagged": bool}, ...],
#          "human_completion": {"text": str, ...} | None,
#          "chosen_completion": int | None   # index into completions actually used
#        },
#        ...
#     ]
#   }
#
# At step i, several *candidate* completions were rated. Only ONE of them (or a
# human-written one, if none were good enough) is chosen to build the prefix for
# step i+1. We build one training example PER CANDIDATE at every step:
#   prefix (built from the chosen path through step i-1) + candidate_text + STEP_TAG
#   -> label token = GOOD_TOKEN if rating==1 else BAD_TOKEN (rating==-1)
# rating==0 ("neutral") is skipped by default (toggle INCLUDE_NEUTRAL_AS_NEGATIVE).

def build_examples(dataset_split):
    examples = []
    for row in dataset_split:
        if row.get("is_quality_control_question") or row.get("is_initial_screening_question"):
            continue  # QC/calibration rows aren't real training signal

        problem = row["question"]["problem"]
        steps = row["label"]["steps"]
        if not steps:
            continue

        prefix_steps: List[str] = []  # text of the chosen path so far

        for step in steps:
            completions = step.get("completions") or []
            for comp in completions:
                if comp.get("flagged"):
                    continue
                rating = comp.get("rating")
                if rating == 1:
                    label = "good"
                elif rating == -1:
                    label = "bad"
                elif rating == 0 and INCLUDE_NEUTRAL_AS_NEGATIVE:
                    label = "bad"
                else:
                    continue  # skip neutral / missing ratings

                solution_so_far = "\n".join(prefix_steps + [comp["text"]])
                text = f"Problem:\n{problem}\n\nSolution:\n{solution_so_far}\n{STEP_TAG}"
                examples.append({"text": text, "label": label})

            # advance the prefix using the chosen completion (or human completion)
            chosen_idx = step.get("chosen_completion")
            if chosen_idx is not None and chosen_idx < len(completions):
                prefix_steps.append(completions[chosen_idx]["text"])
            elif step.get("human_completion") is not None:
                prefix_steps.append(step["human_completion"]["text"])
            else:
                # no way to continue this solution path -> stop building further steps
                break

    return examples

def dedup_examples(examples):
    # PRM800K has multiple labelers who sometimes produce byte-identical
    # (text, label) pairs -- these add zero new signal, just extra compute
    # and extra weight on whatever they happen to say. Drop exact repeats.
    seen = set()
    deduped = []
    for ex in examples:
        key = (ex["text"], ex["label"])
        if key not in seen:
            seen.add(key)
            deduped.append(ex)
    return deduped

train_examples = dedup_examples(build_examples(raw["train"]))
eval_examples = dedup_examples(build_examples(raw["test"]))
print(f"Train examples: {len(train_examples)} | Eval examples: {len(eval_examples)}")
print(train_examples[7]["text"][:500])
print("label:", train_examples[7]["label"])

Train examples: 34054 | Eval examples: 3784
Problem:
How many seconds are in 7.8 minutes?

Solution:
7.8 minutes is the same as 7 minutes and 0.8 minutes.
Right, and since there are 60 seconds in a minute, then there are 60 * 7 = 420 seconds in 7 minutes.
And since there are 60 seconds in a minute, then there are 60 * 0.8 = 48 seconds in 0.8 minutes.
So, in total, there are 420 + 48 = 468 seconds in 7.8 minutes.
That's right!

# Answer

468

Step verdict:
label: good


In [6]:
# %% CELL 5 — Tokenizer setup
# Reuse EXISTING vocabulary tokens for the good/bad targets instead of adding
# brand-new ones. New tokens get random-init embeddings that LoRA (which only
# adapts attention/MLP projections, not embed_tokens/lm_head) never trains --
# that was the bug causing the ~random, ~0.5-probability-for-everything result.
# Reusing pretrained tokens means LoRA only has to *repurpose* an existing,
# meaningful representation, which is far more sample-efficient on ~1k rows.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

CANDIDATE_PAIRS = [(" +", " -"), (" Yes", " No"), ("+", "-"), ("Yes", "No")]
GOOD_ID = BAD_ID = None
for good_str, bad_str in CANDIDATE_PAIRS:
    good_ids = tokenizer.encode(good_str, add_special_tokens=False)
    bad_ids = tokenizer.encode(bad_str, add_special_tokens=False)
    if len(good_ids) == 1 and len(bad_ids) == 1:
        GOOD_ID, BAD_ID = good_ids[0], bad_ids[0]
        print(f"Using existing single-token pair: {good_str!r} -> {GOOD_ID}, "
              f"{bad_str!r} -> {BAD_ID}")
        break
assert GOOD_ID is not None, "None of the candidate pairs tokenized to single ids -- add more candidates"

EOS_ID = tokenizer.eos_token_id

config.json:   0%|          | 0.00/676 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Using existing single-token pair: ' +' -> 488, ' -' -> 481


In [7]:
# %% CELL 6 — Tokenize into (input_ids, labels) pairs, with an ADAPTIVE max length
# labels: -100 everywhere except the final position, which holds GOOD_ID/BAD_ID
# (the model predicts this as the "next token" right after STEP_TAG).
#
# Instead of a fixed guess (1024), measure the real length distribution and
# set MAX_LENGTH to comfortably cover ~95% of examples, rounded to a multiple
# of 32 (efficient for padding/bucketing). This is usually the single biggest
# lever for training speed on this dataset: most step-level judgments are
# short, and padding everything to a rarely-needed worst case wastes a lot of
# compute on every single batch.
probe_lengths = sorted(
    len(tokenizer(e["text"], add_special_tokens=False)["input_ids"]) + 1  # +1 for EOS
    for e in train_examples
)
n = len(probe_lengths)
p50, p90, p95, p_max = (probe_lengths[int(n * q)] if q < 1 else probe_lengths[-1]
                         for q in (0.50, 0.90, 0.95, 1.0))
print(f"token length -> p50={p50} p90={p90} p95={p95} max={p_max}")

MAX_LENGTH = min(MAX_LENGTH_CAP, ((p95 + 31) // 32) * 32)
print(f"Using MAX_LENGTH={MAX_LENGTH} (covers ~95% of examples natively, "
      f"vs the fixed {MAX_LENGTH_CAP} used before)")

def tokenize_example(ex):
    ids = tokenizer(ex["text"], add_special_tokens=False, truncation=True,
                     max_length=MAX_LENGTH - 1)["input_ids"]
    ids = ids + [EOS_ID]  # placeholder final token; its id doesn't matter here
    labels = [-100] * (len(ids) - 1) + [GOOD_ID if ex["label"] == "good" else BAD_ID]
    return {"input_ids": ids, "labels": labels}

def build_tokenized(examples_list):
    tokenized = [tokenize_example(e) for e in examples_list]
    tokenized = [t for t in tokenized if len(t["input_ids"]) <= MAX_LENGTH]
    return tokenized

train_data = build_tokenized(train_examples)
eval_data = build_tokenized(eval_examples)
print(f"Kept after length filter -> train: {len(train_data)}, eval: {len(eval_data)}")

# Wrap in a datasets.Dataset (not just a plain list) so TrainingArguments'
# group_by_length can bucket batches by similar length -- with tens of
# thousands of examples of uneven length (early solution steps are short,
# later ones carry a long accumulated prefix), padding every batch to the
# longest example in a random mix wastes a lot of compute. Sorting into
# length-similar batches avoids that.
from datasets import Dataset as HFDataset
train_data = HFDataset.from_list(train_data)
eval_data = HFDataset.from_list(eval_data)

token length -> p50=368 p90=1153 p95=1442 max=5936
Using MAX_LENGTH=1024 (covers ~95% of examples natively, vs the fixed 1024 used before)
Kept after length filter -> train: 34054, eval: 3784


In [8]:
# %% CELL 7 — Data collator (dynamic padding)
@dataclass
class PRMCollator:
    pad_id: int
 
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        max_len = max(len(f["input_ids"]) for f in features)
        input_ids, attn_mask, labels = [], [], []
        for f in features:
            ids = f["input_ids"]
            lbl = f["labels"]
            pad_len = max_len - len(ids)
            input_ids.append(ids + [self.pad_id] * pad_len)
            attn_mask.append([1] * len(ids) + [0] * pad_len)
            labels.append(lbl + [-100] * pad_len)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attn_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }
 
collator = PRMCollator(pad_id=tokenizer.pad_token_id)
 

In [9]:
# %% CELL 8 — Load Qwen2.5-Math-1.5B + attach LoRA
# No resize_token_embeddings() anymore: we no longer add new tokens (CELL 5),
# so there's nothing untrained to worry about, and the base model's own
# embed_tokens/lm_head stay exactly as pretrained.
#
# IMPORTANT: device_map={"": 0} pins the WHOLE model on GPU 0 instead of
# splitting it across both T4s. Kaggle's T4 pair has no NVLink, so
# device_map="auto" was forcing every forward/backward pass to shuttle
# activations across PCIe at the layer boundary, with only one GPU computing
# at a time -- for a model this small that's pure overhead, no benefit, and
# is also the likely cause of the hangs (flaky cross-GPU P2P/NCCL on
# non-NVLinked cards). A 1.5B model in fp16 (~3GB) fits on a single T4 (16GB)
# with plenty of room left for activations + LoRA, so there's no reason to
# split it at all.
model_kwargs = dict(device_map={"": 0}, trust_remote_code=True, attn_implementation="sdpa")

 
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,   # T4 = Turing, no bf16 support
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs["quantization_config"] = bnb_config
else:
    model_kwargs["dtype"] = torch.float16  # 1.5B fits fine in plain fp16 on a T4
 
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
model.config.pad_token_id = tokenizer.pad_token_id
 
if USE_4BIT:
    model = prepare_model_for_kbit_training(model)
if GRADIENT_CHECKPOINTING and not USE_4BIT:
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
 
lora_config = LoraConfig(
    r=32,           
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
 
# Sanity check BEFORE training: if the base model already separates good/bad
# at all (even weakly), you should see p_good spread out somewhat across a few
# examples here, not stuck at ~0.5 for all of them.
model.eval()
with torch.no_grad():
    for ex in train_examples[:3]:
        ids = tokenizer(ex["text"], add_special_tokens=False,
                         return_tensors="pt").to(model.device)
        logits = model(**ids).logits[0, -1]
        p_good = torch.softmax(torch.stack([logits[GOOD_ID], logits[BAD_ID]]), dim=-1)[0].item()
        print(f"true={ex['label']:>4}  pre-train p_good={p_good:.3f}")
model.train()

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364
true=good  pre-train p_good=0.543
true=good  pre-train p_good=0.924
true=good  pre-train p_good=0.894


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [10]:
 
# %% CELL 8b — (run once per Kaggle account) Hub auth, so training survives session death
# Add a secret named HF_TOKEN via the notebook's Add-ons -> Secrets menu first.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, snapshot_download
 
HUB_MODEL_ID = "aliirtaza7/137qwen25math15b-prm-phase1"
 
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
 
# If a previous (interrupted) run already pushed a checkpoint, pull it down so
# we resume instead of restarting from scratch. Safe to run even if none exists.
#
# IMPORTANT: Trainer's built-in "is this a valid checkpoint" check looks for
# specific weight filenames (model.safetensors / pytorch_model.bin), but PEFT
# (LoRA) saves adapter weights under a different name (adapter_model.safetensors).
# On some transformers versions Trainer doesn't recognize that as valid and
# raises "Can't find a valid checkpoint" even though the adapter weights and
# training state ARE actually there. So: check for the files ourselves first,
# and if full Trainer-level resume looks unreliable, fall back to manually
# loading just the adapter weights (a "warm start" -- you keep the learned
# weights, but the optimizer/scheduler/step-count restart from scratch, which
# is a fine trade for a LoRA run this size rather than losing everything).
resume_path = None
adapter_warm_start_path = None
candidate = f"{OUTPUT_DIR}/last-checkpoint"
try:
    snapshot_download(repo_id=HUB_MODEL_ID, local_dir=OUTPUT_DIR,
                       allow_patterns=["last-checkpoint/*"])
    has_state = os.path.exists(os.path.join(candidate, "trainer_state.json"))
    has_optim = os.path.exists(os.path.join(candidate, "optimizer.pt"))
    has_adapter = os.path.exists(os.path.join(candidate, "adapter_model.safetensors")) or \
                  os.path.exists(os.path.join(candidate, "adapter_model.bin"))
 
    if has_state and has_optim and has_adapter:
        resume_path = candidate
        print(f"Found a complete checkpoint -> attempting full resume from {resume_path}")
    elif has_adapter:
        adapter_warm_start_path = candidate
        print(f"Found adapter weights at {candidate} but the checkpoint is missing "
              f"optimizer/trainer state -- will warm-start weights only (fresh optimizer, "
              f"step count restarts from 0).")
    else:
        print(f"Downloaded {candidate} but it has no adapter weights -- starting fresh.")
except Exception as e:
    print(f"No existing checkpoint found on the Hub, starting fresh. ({e})")
 
if adapter_warm_start_path is not None:
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file as safetensors_load_file
    adapter_file = os.path.join(adapter_warm_start_path, "adapter_model.safetensors")
    if os.path.exists(adapter_file):
        state_dict = safetensors_load_file(adapter_file)
    else:
        state_dict = torch.load(os.path.join(adapter_warm_start_path, "adapter_model.bin"),
                                 map_location="cpu")
    set_peft_model_state_dict(model, state_dict)
    print(f"Warm-started LoRA weights from {adapter_warm_start_path}")
 

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Found a complete checkpoint -> attempting full resume from /kaggle/working/137qwen25math15b-prm-phase1/last-checkpoint


In [12]:
# %% CELL 9 — Training arguments
from transformers import EarlyStoppingCallback
import inspect
 
 
raw_training_args = dict(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,    
    per_device_eval_batch_size=4,       
    gradient_accumulation_steps=4,     
    group_by_length=True,              
    num_train_epochs=2,                
    learning_rate=2e-4,
    weight_decay=0.01,                 
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=300,
    save_strategy="steps",
    save_steps=300,                 
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,                       # T4-friendly (no bf16)
    gradient_checkpointing=GRADIENT_CHECKPOINTING,  # matches CELL 8's model setup
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    report_to="none",
    seed=SEED,
    push_to_hub=True,
    hub_model_id=HUB_MODEL_ID,
    hub_strategy="checkpoint",        # pushes full resumable checkpoint (incl. optimizer state)
)
 
# Defensive against transformers version differences (e.g. some versions use
# "evaluation_strategy" instead of "eval_strategy", or lack a given field
# entirely). Rather than hard-crashing like the group_by_length error you hit,
# silently drop/rename anything unsupported and keep going -- ideally you've
# run CELL 1's upgrade so nothing actually gets dropped here.
valid_params = set(inspect.signature(TrainingArguments.__init__).parameters)
if "eval_strategy" not in valid_params and "evaluation_strategy" in valid_params:
    raw_training_args["evaluation_strategy"] = raw_training_args.pop("eval_strategy")
 
dropped = [k for k in raw_training_args if k not in valid_params]
if dropped:
    hub_related = [k for k in dropped if k in ("push_to_hub", "hub_model_id", "hub_strategy")]
    print(f"WARNING: installed transformers doesn't support: {dropped} -- dropping them and "
          f"continuing. Run CELL 1's pip install + restart the kernel to get full functionality.")
    if hub_related:
        print(f"NOTE: {hub_related} were dropped -- TrainingArguments' automatic Hub push won't "
              f"work on this transformers version. This is fine: CELL 9b below pushes checkpoints "
              f"explicitly via huggingface_hub instead, which doesn't depend on that machinery.")
    for k in dropped:
        raw_training_args.pop(k)
 
training_args = TrainingArguments(**raw_training_args)
 
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [13]:
 # %% CELL 9b — Explicit checkpoint push to the Hub (version-independent)
# Don't rely on TrainingArguments' push_to_hub/hub_strategy 

from transformers import TrainerCallback
from huggingface_hub import HfApi
 
class PushCheckpointToHub(TrainerCallback):
    def __init__(self, repo_id):
        self.api = HfApi()
        self.repo_id = repo_id
 
    def on_save(self, args, state, control, **kwargs):
        ckpt_dir = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")
        if os.path.isdir(ckpt_dir):
            print(f"Pushing {ckpt_dir} -> {self.repo_id}/last-checkpoint ...")
            try:
                self.api.upload_folder(
                    repo_id=self.repo_id,
                    folder_path=ckpt_dir,
                    path_in_repo="last-checkpoint",
                    commit_message=f"checkpoint at step {state.global_step}",
                )
                print("Push complete.")
            except Exception as e:
                print(f"Push failed (training continues locally regardless): {e}")
        return control
 
trainer.add_callback(PushCheckpointToHub(HUB_MODEL_ID))
 

In [15]:
# %% CELL 10 — Train (resumes automatically if CELL 8b found a checkpoint on the Hub)
try:
    trainer.train(resume_from_checkpoint=resume_path)
except ValueError as e:
    if resume_path is not None and "valid checkpoint" in str(e).lower():
        print(f"Trainer rejected the checkpoint at {resume_path} ({e}). "
              f"Falling back to a plain run (any warm-started weights are unaffected).")
        trainer.train(resume_from_checkpoint=None)
    else:
        raise


Step,Training Loss,Validation Loss
2130,0.215216,0.619901


Pushing /kaggle/working/137qwen25math15b-prm-phase1/checkpoint-2130 -> aliirtaza7/137qwen25math15b-prm-phase1/last-checkpoint ...


[transformers] Could not locate the best model at /kaggle/working/137qwen25math15b-prm-phase1/checkpoint-600/pytorch_model.bin, if you are running a distributed training on multiple nodes, you should activate `--save_on_each_node`.


Push complete.


In [16]:
# %% CELL 11 — Save the LoRA adapter (+ tokenizer with the new special tokens)
model.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
print("Saved locally to", f"{OUTPUT_DIR}/final_adapter")
 
from huggingface_hub import HfApi
HfApi().upload_folder(
    repo_id=HUB_MODEL_ID,
    folder_path=f"{OUTPUT_DIR}/final_adapter",
    path_in_repo="final_adapter",
    commit_message="final trained PRM adapter",
)
print(f"Pushed to https://huggingface.co/{HUB_MODEL_ID}/tree/main/final_adapter")


Saved locally to /kaggle/working/137qwen25math15b-prm-phase1/final_adapter


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to https://huggingface.co/aliirtaza7/137qwen25math15b-prm-phase1/tree/main/final_adapter


In [17]:
# %% CELL 13 — Sanity check: Trainer's own eval loss (cross-entropy at tag positions only)
metrics = trainer.evaluate()
print(metrics)
# metrics["eval_loss"] is computed ONLY at the <good_step>/<bad_step> target
# positions (everything else is masked to -100). Falling eval_loss over epochs
# = the model is getting more confident/correct at judging steps. On its own
# it doesn't tell you accuracy though -> see CELL 14 for that.

Training Loss,Validation Loss,Step
0.215216,0.619901,2130


{'eval_loss': 0.6199012994766235}


In [18]:
# %% CELL 14 — Step-level classification metrics on the held-out phase-1 test set
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, confusion_matrix, brier_score_loss,
)

model.eval()

def score_step(text: str) -> float:
    """Returns P(good) for one 'Problem...Solution...\\nStep verdict:' text."""
    ids = tokenizer(text, add_special_tokens=False, truncation=True,
                     max_length=MAX_LENGTH, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**ids).logits[0, -1]  # last position == STEP_TAG
    pair = torch.stack([logits[GOOD_ID], logits[BAD_ID]])
    probs = torch.softmax(pair, dim=-1)
    return probs[0].item()  # P(good)

y_true, y_prob = [], []
for ex in eval_examples:  # from CELL 4 — untokenized text + ground-truth label
    p_good = score_step(ex["text"])
    y_true.append(1 if ex["label"] == "good" else 0)
    y_prob.append(p_good)

y_true = np.array(y_true)
y_prob = np.array(y_prob)
y_pred = (y_prob >= 0.5).astype(int)

acc = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="binary", zero_division=0
)
try:
    auc = roc_auc_score(y_true, y_prob)
except ValueError:
    auc = float("nan")  # only one class present in eval set
brier = brier_score_loss(y_true, y_prob)
cm = confusion_matrix(y_true, y_pred)
majority_baseline = max(y_true.mean(), 1 - y_true.mean())

print(f"n_eval_examples          : {len(y_true)}")
print(f"class balance (%% good)   : {100 * y_true.mean():.1f}%")
print(f"accuracy                 : {acc:.4f}")
print(f"majority-class baseline  : {majority_baseline:.4f}  <- compare accuracy against this")
print(f"precision (good)         : {precision:.4f}")
print(f"recall (good)            : {recall:.4f}")
print(f"f1 (good)                : {f1:.4f}")
print(f"roc_auc                  : {auc:.4f}")
print(f"brier_score              : {brier:.4f}  (lower = better calibrated, 0=perfect, 0.25=uninformative)")
print(f"confusion_matrix [rows=true(bad,good), cols=pred(bad,good)]:\n{cm}")


n_eval_examples          : 3784
class balance (%% good)   : 55.1%
accuracy                 : 0.7518
majority-class baseline  : 0.5513  <- compare accuracy against this
precision (good)         : 0.7369
recall (good)            : 0.8552
f1 (good)                : 0.7917
roc_auc                  : 0.8391
brier_score              : 0.1806  (lower = better calibrated, 0=perfect, 0.25=uninformative)
confusion_matrix [rows=true(bad,good), cols=pred(bad,good)]:
[[1061  637]
 [ 302 1784]]
